# STEP 1. 기상 데이터 수집
기상청 ASOS API → 제주 4개 관측소 → Drive 저장
- 관측소: 제주(184), 서귀포(189), 성산(188), 고산(185)
- 수집 항목: 일 평균기온 / 최고기온 / 최저기온

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import requests
import pandas as pd
from datetime import datetime, timedelta
import os, time

print('✅ 라이브러리 로드 완료')

In [ ]:
# ── 설정 ──────────────────────────────────────────
API_KEY = 'YOUR_API_KEY_HERE'   # ← 기상청 API 키 입력

STATIONS = {184: '제주', 189: '서귀포', 188: '성산', 185: '고산'}

START_YEAR = 2020
END_YEAR   = 2025

SAVE_DIR  = '/content/drive/MyDrive/JADX_병해충/data'
SAVE_FILE = f'{SAVE_DIR}/tb_weather_pest.csv'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ 저장 경로: {SAVE_FILE}')

In [ ]:
def get_asos_daily(stn_id, start_dt, end_dt, api_key):
    url = 'https://apihub.kma.go.kr/api/typ01/url/kma_sfcdd3.php'
    params = {'tm1': start_dt, 'tm2': end_dt, 'stn': stn_id, 'help': 0, 'authKey': api_key}
    try:
        res = requests.get(url, params=params, timeout=10)
        lines = res.text.strip().split('\n')
        rows = []
        for line in lines:
            if line.startswith('#') or not line.strip():
                continue
            cols = line.split()
            if len(cols) < 15:
                continue
            try:
                ta_avg = float(cols[10])
                ta_max = float(cols[11])
                ta_min = float(cols[13])
                if ta_avg <= -90 or ta_max <= -90 or ta_min <= -90:
                    continue
                rows.append({'crtr_ymd': cols[0], 'stn_id': stn_id,
                             'stn_nm': STATIONS.get(stn_id, str(stn_id)),
                             'day_avg_tp': ta_avg, 'day_hghst_tp': ta_max, 'day_lowst_tp': ta_min})
            except (ValueError, IndexError):
                continue
        return pd.DataFrame(rows)
    except Exception as e:
        print(f'    ⚠ API 오류 ({stn_id}, {start_dt}~{end_dt}): {e}')
        return pd.DataFrame()

print('✅ 함수 정의 완료')

In [ ]:
def collect_year(year, api_key):
    all_rows = []
    for month in range(1, 13):
        start = datetime(year, month, 1)
        end = datetime(year, 12, 31) if month == 12 else datetime(year, month+1, 1) - timedelta(days=1)
        for stn_id in STATIONS:
            df = get_asos_daily(stn_id, start.strftime('%Y%m%d'), end.strftime('%Y%m%d'), api_key)
            if not df.empty:
                all_rows.append(df)
            time.sleep(0.2)
    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

all_data = []
for year in range(START_YEAR, END_YEAR + 1):
    print(f'📅 {year}년 수집 중...')
    df_year = collect_year(year, API_KEY)
    if not df_year.empty:
        all_data.append(df_year)
        print(f'  → {len(df_year)}건')

if all_data:
    result = pd.concat(all_data, ignore_index=True).sort_values(['stn_id','crtr_ymd']).reset_index(drop=True)
    result.to_csv(SAVE_FILE, index=False, encoding='utf-8-sig')
    print(f'\n✅ 저장 완료: {len(result)}건 | {result["crtr_ymd"].min()} ~ {result["crtr_ymd"].max()}')

In [ ]:
df = pd.read_csv(SAVE_FILE)
print('[관측소별 수집 건수]')
print(df.groupby('stn_nm')['crtr_ymd'].count().reset_index().rename(columns={'crtr_ymd':'건수'}))
print(f'\n[샘플]')
print(df.head())
print(f'\n[결측값] 평균:{df["day_avg_tp"].isnull().sum()} / 최고:{df["day_hghst_tp"].isnull().sum()} / 최저:{df["day_lowst_tp"].isnull().sum()}')